# Evo2 7B LoRA Fine-Tuning on *N. crassa* OR74A

Fine-tunes Evo2 7B on *Neurospora crassa* OR74A genomic sequences as a proof of concept using LoRA (PEFT).  
Training objective: causal language modeling (next-token prediction) on 8,192 bp windows.

**Requirements:** A100 80GB GPU | HuggingFace token with Evo2 access | Google Drive  
**Colab Secrets required:** `HF_TOKEN`, `GITHUB_PAT` (only needed for the push cell at the end)  
**License:** Apache 2.0

---
| Cell | What it does |
|---|---|
| 1 | Install dependencies |
| 2 | Mount Drive + configure paths and hyperparameters |
| 3 | HuggingFace login |
| 4 | Download *N. crassa* OR74A genome from FungiDB |
| 5 | Prepare genomic windows |
| 6 | Load Evo2 7B and apply LoRA |
| 7 | Train |
| 8 | Save final checkpoint and plot loss |
| 9 | Push notebook to GitHub (optional) |

## Cell 1 — Install dependencies

Run once per runtime. `flash-attn` is installed from a prebuilt wheel — never compile from source on Colab, it takes 30+ minutes and often fails.

In [1]:
import subprocess, sys

FLASH_ATTN_URL = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.1/"
    "flash_attn-2.8.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)

pkgs = [
    "torch torchvision torchaudio",
    "transformers==4.40.2 peft==0.10.0 accelerate",
    "pandas pyarrow numpy",
    "requests biopython matplotlib",
    "evo2",
    FLASH_ATTN_URL,
]

for pkg in pkgs:
    print(f'Installing: {pkg[:60]}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(),
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg[:60]}')
    if result.returncode != 0:
        print(f'  STDERR: {result.stderr[-300:]}')

import torch
print(f'\n✓ PyTorch: {torch.__version__}')
print(f'✓ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Installing: torch torchvision torchaudio...
  ✓ torch torchvision torchaudio
Installing: transformers==4.40.2 peft==0.10.0 accelerate...
  ✓ transformers==4.40.2 peft==0.10.0 accelerate
Installing: pandas pyarrow numpy...
  ✓ pandas pyarrow numpy
Installing: requests biopython matplotlib...
  ✓ requests biopython matplotlib
Installing: evo2...
  ✓ evo2
Installing: https://github.com/Dao-AILab/flash-attention/releases/downlo...
  ✓ https://github.com/Dao-AILab/flash-attention/releases/downlo

✓ PyTorch: 2.10.0+cu128
✓ CUDA available: True
✓ GPU: NVIDIA A100-SXM4-40GB
✓ VRAM: 42.4 GB


## Cell 2 — Mount Drive and configure paths

Edit the paths and hyperparameters here. Everything else in the notebook reads from these variables.

In [2]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────
DRIVE_DIR      = '/content/drive/MyDrive/fungal_evo2'
GENOME_PATH    = f'{DRIVE_DIR}/ncrassa_or74a.fasta'
WINDOWS_PATH   = f'{DRIVE_DIR}/or74a_windows.parquet'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints/lora_run_001'
HF_CACHE_DIR   = f'{DRIVE_DIR}/hf_cache'           # caches Evo2 weights on Drive

for d in [DRIVE_DIR, CHECKPOINT_DIR, HF_CACHE_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Secrets ───────────────────────────────────────────────────────────────
HF_TOKEN    = userdata.get('HF_TOKEN')       # required — Evo2 is gated on HuggingFace
GITHUB_PAT  = userdata.get('GITHUB_PAT')     # only used in the push cell (Cell 9)
GITHUB_REPO = 'Rcperez/fungal_evo2'          # target public repo

os.environ['HF_TOKEN']   = HF_TOKEN
os.environ['HF_HOME']    = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR

# ── LoRA hyperparameters ──────────────────────────────────────────────────
LORA_R       = 8      # rank — increase for more capacity, decrease for fewer params
LORA_ALPHA   = 16     # scaling factor — typically 1x or 2x rank
LORA_DROPOUT = 0.05

# ── Training hyperparameters ──────────────────────────────────────────────
NUM_EPOCHS    = 3
BATCH_SIZE    = 1     # A100 80GB; safe starting point — increase to 2 if VRAM allows
GRAD_ACCUM    = 8     # effective batch size = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY  = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_STEPS  = 100
SAVE_EVERY    = 500   # save checkpoint every N optimizer steps
LOG_EVERY     = 10    # print loss every N optimizer steps
MAX_SEQ_LEN   = 8192  # Evo2 7B context length
SEED          = 42

print('Configuration:')
print(f'  Drive dir:      {DRIVE_DIR}')
print(f'  Genome:         {GENOME_PATH}')
print(f'  Windows:        {WINDOWS_PATH}')
print(f'  Checkpoints:    {CHECKPOINT_DIR}')
print(f'  HF cache:       {HF_CACHE_DIR}')
print(f'  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'  Epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}')
print(f'  LR={LEARNING_RATE}, warmup={WARMUP_STEPS} steps')
print(f'  HF_TOKEN set: {bool(HF_TOKEN)}')

Mounted at /content/drive
Configuration:
  Drive dir:      /content/drive/MyDrive/fungal_evo2
  Genome:         /content/drive/MyDrive/fungal_evo2/ncrassa_or74a.fasta
  Windows:        /content/drive/MyDrive/fungal_evo2/or74a_windows.parquet
  Checkpoints:    /content/drive/MyDrive/fungal_evo2/checkpoints/lora_run_001
  HF cache:       /content/drive/MyDrive/fungal_evo2/hf_cache
  LoRA r=8, alpha=16, dropout=0.05
  Epochs=3, batch=1, grad_accum=8
  LR=0.0002, warmup=100 steps
  HF_TOKEN set: True


## Cell 3 — HuggingFace login

Evo2 7B weights are gated on HuggingFace. You must accept the model terms at  
https://huggingface.co/arcinstitute/evo2_7b_base before this will work.

In [3]:
import subprocess

result = subprocess.run(
    ['huggingface-cli', 'login', '--token', HF_TOKEN, '--add-to-git-credential'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✓ HuggingFace login successful')
else:
    print('✗ HuggingFace login failed')
    print(result.stderr)
    raise RuntimeError(
        'HuggingFace login failed. Check that HF_TOKEN is set in Colab Secrets '
        'and that you have accepted the Evo2 model terms at '
        'https://huggingface.co/arcinstitute/evo2_7b_base'
    )

✓ HuggingFace login successful


## Cell 4 — Download *N. crassa* OR74A genome

Source: FungiDB release 68. Public download, no authentication required.  
~41 MB. Skipped automatically if the file already exists on Drive.

In [4]:
import requests
from pathlib import Path

FUNGIDB_GENOME_URL = (
    'https://fungidb.org/common/downloads/release-68/NcrassaOR74A/'
    'fasta/data/FungiDB-68_NcrassaOR74A_Genome.fasta'
)

genome_path = Path(GENOME_PATH)

if genome_path.exists():
    size_mb = genome_path.stat().st_size / 1e6
    print(f'✓ Genome already on Drive ({size_mb:.1f} MB) — skipping download')
else:
    print(f'Downloading OR74A genome from FungiDB release 68...')
    response = requests.get(FUNGIDB_GENOME_URL, stream=True, timeout=180)
    response.raise_for_status()
    genome_path.parent.mkdir(parents=True, exist_ok=True)
    bytes_written = 0
    with open(genome_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            bytes_written += len(chunk)
            if bytes_written % (10 * 1024 * 1024) < 1024 * 1024:
                print(f'  {bytes_written / 1e6:.0f} MB...')
    size_mb = genome_path.stat().st_size / 1e6
    print(f'✓ Downloaded: {size_mb:.1f} MB')

# Sanity check — confirm 7 chromosomes present
with open(genome_path) as f:
    headers = [l.strip()[1:].split()[0] for l in f if l.startswith('>')]

OR74A_CHROMOSOMES = {'CM002236','CM002237','CM002238','CM002239','CM002240','CM002241','CM002242'}
found_chroms = set(headers) & OR74A_CHROMOSOMES

print(f'  Total sequences in FASTA: {len(headers)}')
print(f'  OR74A chromosomes found:  {len(found_chroms)}/7')

if len(found_chroms) < 7:
    missing = OR74A_CHROMOSOMES - found_chroms
    raise RuntimeError(f'Missing chromosomes: {missing}. Check genome download.')

print('✓ Genome validated')

  10 MB...
  21 MB...
  31 MB...
✓ Downloaded: 41.8 MB
  Total sequences in FASTA: 21
  OR74A chromosomes found:  7/7
✓ Genome validated


## Cell 5 — Prepare genomic windows

Partitions each chromosome into non-overlapping 8,192 bp windows (Evo2 7B context length).  
Windows with >10% N characters are excluded.  
Incomplete windows at chromosome ends are discarded.  
Output saved as parquet to Drive. Skipped if already present.

In [5]:
import pandas as pd
from pathlib import Path

# OR74A 7 chromosomes (FungiDB accessions)
# Excludes 13 unplaced scaffolds (KI440765-KI440777) and mitochondrial genome
OR74A_CHROMOSOMES = {
    'CM002236', 'CM002237', 'CM002238',
    'CM002239', 'CM002240', 'CM002241', 'CM002242',
}
WINDOW_SIZE    = 8_192
MAX_N_FRACTION = 0.10


def parse_fasta(fasta_path):
    """Parse a FASTA file into {sequence_name: sequence_string} dict."""
    sequences    = {}
    current_name = None
    current_seq  = []
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_name is not None:
                    sequences[current_name] = ''.join(current_seq).upper()
                current_name = line[1:].split()[0]
                current_seq  = []
            else:
                current_seq.append(line)
        if current_name is not None:
            sequences[current_name] = ''.join(current_seq).upper()
    return sequences


def n_fraction(sequence):
    """Return fraction of N characters in a sequence string."""
    if not sequence:
        return 1.0
    return sequence.count('N') / len(sequence)


def generate_windows(fasta_path, window_size=WINDOW_SIZE,
                     max_n_fraction=MAX_N_FRACTION, chromosomes=None):
    """Generate non-overlapping windows from the OR74A genome FASTA.

    Args:
        fasta_path:      Path to genome FASTA file.
        window_size:     Window size in bp. Default 8,192 (Evo2 context length).
        max_n_fraction:  Max fraction of N characters allowed. Default 0.10.
        chromosomes:     Set of chromosome accessions to include.

    Returns:
        pandas DataFrame with columns: window_id, chromosome, start, end, sequence.
    """
    if chromosomes is None:
        chromosomes = OR74A_CHROMOSOMES

    print('Parsing genome FASTA...')
    sequences = parse_fasta(fasta_path)
    print(f'  {len(sequences)} sequences loaded')

    records   = []
    window_id = 0

    for chrom in sorted(chromosomes):
        if chrom not in sequences:
            print(f'  Warning: {chrom} not in FASTA — skipping')
            continue

        seq          = sequences[chrom]
        n_complete   = len(seq) // window_size
        n_excluded   = 0

        for i in range(n_complete):
            start      = i * window_size
            end        = start + window_size
            window_seq = seq[start:end]

            if n_fraction(window_seq) > max_n_fraction:
                n_excluded += 1
                continue

            records.append({
                'window_id':  window_id,
                'chromosome': chrom,
                'start':      start,
                'end':        end,
                'sequence':   window_seq,
            })
            window_id += 1

        print(f'  {chrom}: {n_complete} windows, {n_excluded} excluded (N > {max_n_fraction:.0%})')

    df = pd.DataFrame(records)
    print(f'\nTotal windows: {len(df):,}')
    return df


windows_path = Path(WINDOWS_PATH)

if windows_path.exists():
    df = pd.read_parquet(windows_path)
    print(f'✓ Windows already on Drive — loaded {len(df):,} windows')
else:
    df = generate_windows(GENOME_PATH)
    windows_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(windows_path, index=False)
    size_mb = windows_path.stat().st_size / 1e6
    print(f'✓ Saved {len(df):,} windows to Drive ({size_mb:.1f} MB)')

print(f'\nSample:')
print(df[['window_id','chromosome','start','end']].head())

Parsing genome FASTA...
  21 sequences loaded
  CM002236: 1196 windows, 1 excluded (N > 10%)
  CM002237: 546 windows, 0 excluded (N > 10%)
  CM002238: 643 windows, 0 excluded (N > 10%)
  CM002239: 732 windows, 0 excluded (N > 10%)
  CM002240: 785 windows, 0 excluded (N > 10%)
  CM002241: 514 windows, 0 excluded (N > 10%)
  CM002242: 519 windows, 0 excluded (N > 10%)

Total windows: 4,934
✓ Saved 4,934 windows to Drive (18.9 MB)

Sample:
   window_id chromosome  start    end
0          0   CM002236      0   8192
1          1   CM002236   8192  16384
2          2   CM002236  16384  24576
3          3   CM002236  24576  32768
4          4   CM002236  32768  40960


## Cell 6 — Load Evo2 7B and apply LoRA

Wraps the StripedHyena (Vortex) base model with LoRA adapters on the 5 attention blocks  
(positions 3, 10, 17, 24, 31). All base model parameters are frozen.  
Only LoRA parameters (~983K of 6.48B total) are trained.

In [ ]:
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType

# ── Load Evo2 7B ──────────────────────────────────────────────────────────
print('Loading Evo2 7B base model (this may take a few minutes)...')
from evo2 import Evo2

evo2_obj   = Evo2('evo2_7b_base')
base_model = evo2_obj.model       # raw StripedHyena PyTorch module
tokenizer  = evo2_obj.tokenizer

print('✓ Evo2 7B loaded')
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM used: {allocated:.1f} / {total:.1f} GB')

# ── Apply LoRA ────────────────────────────────────────────────────────────
# Evo2 7B (StripedHyena) has 32 blocks: 27 Hyena SSM blocks + 5 attention blocks.
# LoRA targets the attention blocks only (positions 3, 10, 17, 24, 31).
# Hyena TELinear layers are TransformerEngine-specific and not PEFT-compatible.
ATTENTION_BLOCK_POSITIONS = [3, 10, 17, 24, 31]
target_modules = []
for idx in ATTENTION_BLOCK_POSITIONS:
    target_modules += [
        f'blocks.{idx}.inner_mha_cls.Wqkv',
        f'blocks.{idx}.inner_mha_cls.out_proj',
    ]

# Compatibility patch: Vortex stores model config as a plain namespace.
# PEFT requires model.config to have a to_dict() method.
if hasattr(base_model, 'config'):
    cfg = base_model.config
    if not hasattr(cfg, 'to_dict') or cfg.to_dict is None:
        cfg.to_dict = lambda: vars(cfg) if hasattr(cfg, '__dict__') else {}
    if not hasattr(cfg, 'model_type'):
        cfg.model_type = 'stripedhyena'

peft_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = target_modules,
    bias           = 'none',
    task_type      = TaskType.FEATURE_EXTRACTION,
)

lora_model = get_peft_model(base_model, peft_config)

# Compatibility patch: Vortex loads some parameters in inference mode,
# which blocks autograd. Clone them to normal mode so gradients flow.
converted = 0
for name, param in lora_model.named_parameters():
    if param.is_inference():
        normal = param.detach().clone().requires_grad_(param.requires_grad)
        parts  = name.split('.')
        parent = lora_model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], nn.Parameter(normal, requires_grad=param.requires_grad))
        converted += 1

print(f'  Inference-mode parameters converted: {converted}')
lora_model.print_trainable_parameters()
print('✓ LoRA applied')

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading Evo2 7B base model (this may take a few minutes)...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

evo2_7b_base.pt:   0%|          | 0.00/13.0G [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

Found complete file in repo: evo2_7b_base.pt


/usr/local/lib/python3.12/dist-packages/evo2/models.py:282: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(
100%|██████████| 32/32 [00:00<00:00, 90.34it/s]


Extra keys in state_dict: {'blocks.26.projections._extra_state', 'blocks.17.mixer.dense._extra_state', 'blocks.15.projections._extra_state', 'blocks.21.projections._extra_state', 'blocks.18.projections._extra_state', 'blocks.24.mixer.dense._extra_state', 'blocks.28.projections._extra_state', 'blocks.16.projections._extra_state', 'blocks.2.projections._extra_state', 'blocks.29.projections._extra_state', 'blocks.27.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.30.projections._extra_state', 'blocks.14.projections._extra_state', 'blocks.7.projections._extra_state', 'blocks.23.projections._extra_state', 'blocks.10.mixer.dense._extra_state', 'blocks.5.projections._extra_state', 'blocks.3.mixer.dense._extra_state', 'blocks.1.projections._extra_state', 'unembed.weight', 'blocks.8.projections._extra_state', 'blocks.6.projections._extra_state', 'blocks.4.projections._extra_state', 'blocks.9.projections._extra_state', 'blocks.20.projections._extra_state', 'blocks.31.mix

## Cell 7 — Train

Standard causal language modeling loop. Estimated ~55 min on a single A100 80GB for 3 epochs over ~4,900 windows.

In [ ]:
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# ── Reproducibility ───────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Dataset ───────────────────────────────────────────────────────────────
class GenomicWindowDataset(Dataset):
    """Dataset of fixed-length genomic windows, tokenized for Evo2.

    Reads the 'sequence' column from the parquet file produced in Cell 5.
    Tokenization uses Evo2's character-level tokenizer.
    """
    def __init__(self, parquet_path, tokenizer, max_seq_len=8192):
        self.tokenizer   = tokenizer
        self.max_seq_len = max_seq_len
        df = pd.read_parquet(parquet_path)
        if 'sequence' not in df.columns:
            raise ValueError(f'Parquet must have a sequence column. Found: {list(df.columns)}')
        self.sequences = df['sequence'].tolist()
        print(f'  Dataset: {len(self.sequences):,} windows')

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq    = self.sequences[idx][:self.max_seq_len]
        tokens = self.tokenizer.tokenize(seq)
        return torch.tensor(tokens, dtype=torch.int)


def collate_fn(batch):
    """Pad sequences in a batch to the same length with token 0."""
    max_len = max(t.shape[0] for t in batch)
    padded  = torch.zeros(len(batch), max_len, dtype=torch.int)
    for i, t in enumerate(batch):
        padded[i, :t.shape[0]] = t
    return padded


# ── Loss ──────────────────────────────────────────────────────────────────
def causal_lm_loss(logits, input_ids):
    """Next-token prediction cross-entropy loss."""
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous().long()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=0,
    )


# ── LR schedule ───────────────────────────────────────────────────────────
def get_lr(step, warmup_steps, total_steps, max_lr):
    """Linear warmup followed by cosine decay."""
    if step < warmup_steps:
        return max_lr * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


# ── Checkpoint ────────────────────────────────────────────────────────────
def save_checkpoint(model, optimizer, step, epoch, loss):
    ckpt_dir = Path(CHECKPOINT_DIR) / f'step_{step:06d}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ckpt_dir))  # LoRA adapter weights only (~4 MB for r=8)
    torch.save({
        'step': step, 'epoch': epoch, 'loss': loss,
        'optimizer_state_dict': optimizer.state_dict(),
    }, ckpt_dir / 'trainer_state.pt')
    print(f'  ✓ Checkpoint -> {ckpt_dir}')


# ── Build dataset and dataloader ──────────────────────────────────────────
print('Building dataset...')
dataset    = GenomicWindowDataset(WINDOWS_PATH, tokenizer, max_seq_len=MAX_SEQ_LEN)
dataloader = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    collate_fn  = collate_fn,
    num_workers = 2,
    pin_memory  = True,
)
total_steps = len(dataloader) * NUM_EPOCHS // GRAD_ACCUM
print(f'  {len(dataloader):,} batches/epoch  |  {total_steps:,} total optimizer steps')

# ── Optimizer ─────────────────────────────────────────────────────────────
lora_params = [p for p in lora_model.parameters() if p.requires_grad]
optimizer   = torch.optim.AdamW(lora_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ── Training loop ─────────────────────────────────────────────────────────
lora_model.train()
global_step = 0
history     = {'steps': [], 'losses': [], 'lrs': []}
t0          = time.time()

print(f'\nStarting training — {NUM_EPOCHS} epochs  |  '
      f'effective batch size {BATCH_SIZE * GRAD_ACCUM}\n')

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, input_ids in enumerate(dataloader):
        input_ids = input_ids.to('cuda:0')

        with torch.inference_mode(False):
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                output = lora_model.base_model.model(input_ids)
                logits = output[0] if isinstance(output, tuple) else output
                loss   = causal_lm_loss(logits.float(), input_ids) / GRAD_ACCUM

        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            lr = get_lr(global_step, WARMUP_STEPS, total_steps, LEARNING_RATE)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            torch.nn.utils.clip_grad_norm_(lora_params, MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOG_EVERY == 0:
                avg   = epoch_loss / (batch_idx + 1)
                vram  = torch.cuda.memory_allocated() / 1e9
                mins  = (time.time() - t0) / 60
                print(f'Ep {epoch+1}/{NUM_EPOCHS} | '
                      f'Step {global_step:>5}/{total_steps} | '
                      f'Loss {avg:.4f} | LR {lr:.2e} | '
                      f'VRAM {vram:.1f}GB | {mins:.1f}min')
                history['steps'].append(global_step)
                history['losses'].append(avg)
                history['lrs'].append(lr)

            if global_step % SAVE_EVERY == 0:
                save_checkpoint(lora_model, optimizer, global_step, epoch,
                                epoch_loss / (batch_idx + 1))

    avg_ep = epoch_loss / len(dataloader)
    print(f'\nEpoch {epoch+1} done | Avg loss: {avg_ep:.4f}\n')

elapsed = (time.time() - t0) / 60
print(f'✓ Training complete — {global_step} steps in {elapsed:.1f} min')

Building dataset...
  Dataset: 4,934 windows
  4,934 batches/epoch  |  1,850 total optimizer steps

Starting training — 3 epochs  |  effective batch size 8



OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 39.49 GiB of which 1.92 GiB is free. Including non-PyTorch memory, this process has 37.56 GiB memory in use. Of the allocated memory 34.78 GiB is allocated by PyTorch, and 2.28 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Cell 8 — Save final checkpoint and plot loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

final_dir = Path(CHECKPOINT_DIR) / 'final'
final_dir.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter weights
lora_model.save_pretrained(str(final_dir))
np.save(final_dir / 'loss_history.npy', np.array(history['losses']))
np.save(final_dir / 'lr_history.npy',   np.array(history['lrs']))

files = [f.name for f in final_dir.iterdir()]
print(f'✓ Final checkpoint saved to {final_dir}')
print(f'  Files: {files}')
if history['losses']:
    print(f'  Final loss: {history["losses"][-1]:.4f}')

# Loss curve
if history['losses']:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['steps'], history['losses'], linewidth=1.5)
    ax.set_xlabel('Optimizer step')
    ax.set_ylabel('Loss')
    ax.set_title('N. crassa OR74A — Evo2 7B LoRA Training Loss')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(final_dir / 'loss_curve.png'), dpi=120)
    plt.show()
    print('✓ Loss curve saved')

## Cell 9 — Push notebook to GitHub (optional)

Copies this notebook from Drive to the local Colab environment and pushes it to  
`Rcperez/fungal_evo2`. Requires `GITHUB_PAT` in Colab Secrets with `repo` scope.

Only run this cell when you want to publish an updated version of the notebook.

In [ ]:
# ── Cell 9 — Push all repo files to GitHub ───────────────────────────────
#
# Expects the following files to be uploaded to this Colab session:
#   /content/README.md
#   /content/LICENSE
#   /content/requirements.txt
#   /content/.gitignore
#
# The notebook itself is read from Drive (where you saved it).
# Requires GITHUB_PAT in Colab Secrets with repo scope.

import subprocess
import shutil
from pathlib import Path

REPO_NAME    = 'fungal_evo2'
CLONE_DIR    = f'/content/{REPO_NAME}'
NOTEBOOK_SRC = '/content/drive/MyDrive/fungal_evo2/ncrassa_evo2_finetune.ipynb'

# Files uploaded directly to this Colab session
UPLOADED_FILES = [
    '/content/README.md',
    '/content/LICENSE',
    '/content/requirements.txt',
    '/content/.gitignore',
]

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd,
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}\n{result.stderr}')
    return result.stdout.strip()

# ── Clone or pull ─────────────────────────────────────────────────────────
if Path(CLONE_DIR).exists():
    print('Repo already cloned — pulling latest...')
    run('git pull', cwd=CLONE_DIR)
else:
    print('Cloning repo...')
    run(f'git clone https://{GITHUB_PAT}@github.com/{GITHUB_REPO}.git {CLONE_DIR}')

print(f'✓ Repo ready at {CLONE_DIR}')

# ── Copy notebook from Drive ──────────────────────────────────────────────
nb_src = Path(NOTEBOOK_SRC)
nb_dst = Path(CLONE_DIR) / 'ncrassa_evo2_finetune.ipynb'

if not nb_src.exists():
    raise FileNotFoundError(
        f'Notebook not found at {nb_src}. '
        'Save this notebook to Drive first (File -> Save a copy in Drive), '
        f'then confirm it is at {NOTEBOOK_SRC}'
    )

shutil.copy2(nb_src, nb_dst)
print(f'✓ Notebook copied from Drive')

# ── Copy uploaded files ───────────────────────────────────────────────────
for src_path in UPLOADED_FILES:
    src = Path(src_path)
    if not src.exists():
        raise FileNotFoundError(
            f'Expected uploaded file not found: {src_path}\n'
            'Upload all four files (README.md, LICENSE, requirements.txt, .gitignore) '
            'using the Colab file panel before running this cell.'
        )
    dst = Path(CLONE_DIR) / src.name
    shutil.copy2(src, dst)
    print(f'✓ Copied: {src.name}')

# ── Git config ────────────────────────────────────────────────────────────
run('git config user.email "rcperez@stanford.edu"', cwd=CLONE_DIR)
run('git config user.name "Rolando Perez"', cwd=CLONE_DIR)

# ── Stage all, commit, push ───────────────────────────────────────────────
run('git add ncrassa_evo2_finetune.ipynb README.md LICENSE requirements.txt .gitignore',
    cwd=CLONE_DIR)

status = run('git status --porcelain', cwd=CLONE_DIR)
if not status:
    print('\nNo changes to commit — all files already up to date.')
else:
    print(f'\nChanges staged:\n{status}')
    run('git commit -m "Add Evo2 LoRA fine-tuning notebook and repo files"', cwd=CLONE_DIR)
    run('git push', cwd=CLONE_DIR)
    print(f'\n✓ All files pushed to https://github.com/{GITHUB_REPO}')